# 05 – Validation and inference (Day 10)

**Scope:** validate all 12,870 2026 rows against the same-month 2025 index, then answer three pre-specified state-level questions. The state set is the full set represented in this file, not a random sample. Treat p-values as descriptive diagnostics, not population-level evidence.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
cpi = pd.read_csv(ROOT / "data/cleaned/cpi_clean.csv")
months = ["January", "February", "March", "April", "May", "June", "July", "August", "September"]
df = cpi[(cpi["year"] == 2026) & (cpi["month_num"] <= 9)].copy()
states = df[~df["is_aggregate"]].copy()
print("2026 observations:", len(cpi[cpi.year == 2026]))

2026 observations: 12870


## Part A – validation first

### 1. Recompute year-on-year inflation from the index

Formula: `(current_month_index / same_month_previous_year_index - 1) * 100`. The published values are rounded to two decimals, so tiny differences are expected.

In [2]:
prior = cpi[cpi["year"] == 2025][["month", "month_num", "state", "sector", "division", "cpi_index"]].rename(columns={"cpi_index": "prior_year_index"})
current = cpi[cpi["year"] == 2026].merge(prior, on=["month", "month_num", "state", "sector", "division"], how="left", validate="one_to_one")
current["recomputed_yoy"] = (current["cpi_index"] / current["prior_year_index"] - 1) * 100
current["difference_pp"] = current["recomputed_yoy"] - current["inflation"]
validation_summary = current["difference_pp"].abs().agg(["count", "mean", "median", "max"])
print("Missing prior-year index:", current["prior_year_index"].isna().sum())
print("Missing published inflation:", current["inflation"].isna().sum())
display(validation_summary.to_frame("absolute difference (pp)"))
print("All differences <= 0.02 percentage points:", bool((current["difference_pp"].abs() <= 0.02).all()))
display(current.loc[current["difference_pp"].abs().nlargest(10).index,
                     ["month", "state", "sector", "division", "recomputed_yoy", "inflation", "difference_pp"]])

Missing prior-year index: 0
Missing published inflation: 0


,absolute difference (pp)
count,12870.000000
mean,0.004011
median,0.003466
max,0.014142


All differences <= 0.02 percentage points: True


,month,state,sector,division,recomputed_yoy,inflation,difference_pp
12550,May,West Bengal,Combined,Food and beverages,4.664142,4.65,0.014142
7455,April,Meghalaya,Rural,"Personal care, social protection and miscellan...",3.455997,3.47,-0.014003
8303,June,NCT of Delhi,Urban,Transport,2.443890,2.43,0.013890
7074,January,Manipur,Rural,"Housing, water, electricity, gas and other fuels",2.756141,2.77,-0.013859
12320,September,Uttarakhand,Rural,Food and beverages,8.086202,8.10,-0.013798
12232,February,Uttarakhand,Combined,Information and communication,-1.096236,-1.11,0.013764
555,July,Andaman And Nicobar Islands,Rural,"Personal care, social protection and miscellan...",19.736264,19.75,-0.013736
11704,May,Tripura,Urban,CPI (General),2.813681,2.80,0.013681
5277,April,Kerala,Rural,Clothing and footwear,0.383594,0.37,0.013594
6282,January,Madhya Pradesh,Combined,"Personal care, social protection and miscellan...",20.333528,20.32,0.013528


### 2. Check whether Combined index lies between Rural and Urban

The check is performed for every 2026 month, state, and division with all three sectors present. Chandigarh is expected to be absent because it has no Rural observations.

In [3]:
wide = cpi[(cpi["year"] == 2026) & (~cpi["is_aggregate"])].pivot(index=["month", "month_num", "state", "division"], columns="sector", values="cpi_index").reset_index()
complete = wide.dropna(subset=["Rural", "Urban", "Combined"]).copy()
complete["lower_ru"] = complete[["Rural", "Urban"]].min(axis=1)
complete["upper_ru"] = complete[["Rural", "Urban"]].max(axis=1)
complete["combined_between_ru"] = complete["Combined"].between(complete["lower_ru"] - 1e-10, complete["upper_ru"] + 1e-10)
print("Complete comparisons:", len(complete))
print("Combined outside Rural–Urban interval:", int((~complete["combined_between_ru"]).sum()))
print("Chandigarh Rural rows:", len(cpi[(cpi.year == 2026) & (cpi.state == "Chandigarh") & (cpi.sector == "Rural")]))
assert complete["combined_between_ru"].all()

Complete comparisons: 4095
Combined outside Rural–Urban interval: 0
Chandigarh Rural rows: 0


### 3. External headline cross-check

The dataset's All India / Combined / CPI (General) September 2026 value is compared with the Ministry of Statistics and Programme Implementation (MoSPI) press release. The October 2026 release reports September as final.

In [4]:
headline = cpi[(cpi.year == 2026) & (cpi.month == "September") & (cpi.state == "All India") &
                (cpi.sector == "Combined") & (cpi.division == "CPI (General)")]["inflation"].iloc[0]
official_value = 5.54
print(f"Project data: {headline:.2f}% | MoSPI published: {official_value:.2f}% | Difference: {headline-official_value:.2f} pp")
assert round(headline, 2) == official_value

Project data: 5.54% | MoSPI published: 5.54% | Difference: 0.00 pp


## Part B – pre-specified inference questions

1. **September rural–urban difference:** Is Rural CPI (General) inflation different from Urban inflation across states? Paired observations; Chandigarh excluded because Rural is unavailable.
2. **January–September change:** Did Combined CPI (General) inflation rise between January and September across states? Paired observations.
3. **Food/general association:** Across states in September, is Combined Food and beverages inflation associated with Combined CPI (General) inflation? Spearman rank correlation.

For paired differences, inspect Shapiro–Wilk normality before choosing the paired test. Effect sizes and confidence intervals are reported alongside p-values.

### Question 1 – Rural versus Urban, September 2026

In [5]:
ru = states[(states["month"] == "September") & (states["division"] == "CPI (General)") &
            (states["sector"].isin(["Rural", "Urban"]))].pivot(index="state", columns="sector", values="inflation").dropna()
ru_diff = ru["Rural"] - ru["Urban"]
normality_ru = stats.shapiro(ru_diff)
print("n paired states:", len(ru_diff), "; Shapiro p:", normality_ru.pvalue)
if normality_ru.pvalue >= 0.05:
    test_ru = stats.ttest_rel(ru["Rural"], ru["Urban"])
    chosen_ru = "paired t-test"
else:
    test_ru = stats.wilcoxon(ru["Rural"], ru["Urban"], alternative="two-sided")
    chosen_ru = "Wilcoxon signed-rank test"
mean_diff = ru_diff.mean(); median_diff = ru_diff.median()
se = stats.sem(ru_diff); crit = stats.t.ppf(0.975, len(ru_diff)-1)
mean_ci = (mean_diff-crit*se, mean_diff+crit*se)
rng = np.random.default_rng(20261120)
boot_medians = np.array([np.median(rng.choice(ru_diff.to_numpy(), len(ru_diff), replace=True)) for _ in range(20000)])
median_ci = np.quantile(boot_medians, [0.025, 0.975])
print("Chosen test:", chosen_ru, "; statistic:", test_ru.statistic, "; p:", test_ru.pvalue)
print("Rural - Urban mean difference (pp):", mean_diff, "; 95% t CI:", mean_ci)
print("Median difference (pp):", median_diff, "; 95% bootstrap percentile CI:", median_ci)

n paired states: 35 ; Shapiro p: 0.03185540489601302


Chosen test: Wilcoxon signed-rank test ; statistic: 45.0 ; p: 9.759121043234408e-06
Rural - Urban mean difference (pp): 0.9102857142857143 ; 95% t CI: (np.float64(0.6209003160117728), np.float64(1.1996711125596558))
Median difference (pp): 1.0 ; 95% bootstrap percentile CI: [0.64 1.45]


### Question 2 – January versus September, Combined CPI (General)

In [6]:
jan = states[(states["month"] == "January") & (states["sector"] == "Combined") & (states["division"] == "CPI (General)")].set_index("state")["inflation"]
sep = states[(states["month"] == "September") & (states["sector"] == "Combined") & (states["division"] == "CPI (General)")].set_index("state")["inflation"]
paired = pd.concat([jan.rename("January"), sep.rename("September")], axis=1).dropna()
diff = paired["September"] - paired["January"]
normality_js = stats.shapiro(diff)
print("n paired states:", len(diff), "; Shapiro p:", normality_js.pvalue)
if normality_js.pvalue >= 0.05:
    test_js = stats.ttest_rel(paired["September"], paired["January"])
    chosen_js = "paired t-test"
else:
    test_js = stats.wilcoxon(paired["September"], paired["January"], alternative="two-sided")
    chosen_js = "Wilcoxon signed-rank test"
mean_diff = diff.mean(); median_diff = diff.median(); se = stats.sem(diff); crit = stats.t.ppf(0.975, len(diff)-1)
mean_ci = (mean_diff-crit*se, mean_diff+crit*se)
print("Chosen test:", chosen_js, "; statistic:", test_js.statistic, "; p:", test_js.pvalue)
print("September - January mean difference (pp):", mean_diff, "; 95% CI:", mean_ci)
print("Median difference (pp):", median_diff)

n paired states: 36 ; Shapiro p: 0.9702835354333403
Chosen test: paired t-test ; statistic: 13.832416517167637 ; p: 9.42260068312711e-16
September - January mean difference (pp): 2.6269444444444443 ; 95% CI: (np.float64(2.24140221202822), np.float64(3.0124866768606684))
Median difference (pp): 2.635


### Question 3 – Food inflation and general inflation across states

In [7]:
food = states[(states["month"] == "September") & (states["sector"] == "Combined") &
               (states["division"] == "Food and beverages")].set_index("state")["inflation"]
general = states[(states["month"] == "September") & (states["sector"] == "Combined") &
                 (states["division"] == "CPI (General)")].set_index("state")["inflation"]
food_general = pd.concat([food.rename("Food"), general.rename("General")], axis=1).dropna()
spearman = stats.spearmanr(food_general["Food"], food_general["General"])
rng = np.random.default_rng(20261120); boot_rho = []
for _ in range(20000):
    idx = rng.integers(0, len(food_general), len(food_general))
    boot_rho.append(stats.spearmanr(food_general["Food"].to_numpy()[idx], food_general["General"].to_numpy()[idx]).statistic)
rho_ci = np.nanquantile(boot_rho, [0.025, 0.975])
print("n states:", len(food_general), "; Spearman rho:", spearman.statistic, "; p:", spearman.pvalue)
print("95% bootstrap percentile CI for rho:", rho_ci)

n states: 36 ; Spearman rho: 0.824301718760548 ; p: 6.472724350777418e-10
95% bootstrap percentile CI for rho: [0.62817376 0.92750384]


## Methods and caveats

- The 36 state/UT observations are the full set available in this file, not a random sample. P-values are descriptive diagnostics and should not be interpreted as population-sampling evidence.
- The file does not expose the underlying price-quote sampling uncertainty, collection design, or price-level microdata. The true measurement uncertainty is therefore not estimated here.
- Neighbouring states may share price shocks; state observations are not fully independent. The tests do not model spatial dependence.
- State averages are unweighted. They are not substitutes for the official weighted All India CPI.
- The normality check is a practical diagnostic, not proof of a distribution. The first paired difference is non-normal at the 5% threshold, so the Wilcoxon signed-rank test is used. The January–September differences pass the Shapiro–Wilk check, so a paired t-test is used.
- Confidence intervals are conventional paired t intervals for mean differences; percentile bootstrap intervals are used for the median difference and Spearman rho. Results are descriptive and should be read with the caveats above.

In [8]:
(ROOT / "docs").mkdir(exist_ok=True)
current[["month", "state", "sector", "division", "cpi_index", "prior_year_index", "recomputed_yoy", "inflation", "difference_pp"]].to_csv(ROOT / "report/yoy_validation_all_2026.csv", index=False)
print("Validation output exported to report/yoy_validation_all_2026.csv")

Validation output exported to report/yoy_validation_all_2026.csv
